In [1]:
# !pip install transformers datasets numpy torch accelerate -q

In [2]:
from datasets import load_dataset

In [3]:
papers_data = load_dataset('csv', data_files='./data/redacoes.csv')

In [4]:
papers_data

DatasetDict({
    train: Dataset({
        features: ['essay', 'score'],
        num_rows: 4570
    })
})

In [5]:
train_test = papers_data['train'].train_test_split(test_size=0.2, shuffle=False)

In [6]:
from datasets import DatasetDict

In [7]:
# base_model = 'Geotrend/distilbert-base-pt-cased'
base_model = 'Mel-Iza0/zero-shot'

In [8]:
from transformers import AutoTokenizer
tokenizer = AutoTokenizer.from_pretrained(base_model)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


In [9]:
def tokenizer_data(text_data):
    return tokenizer(text_data['essay'], truncation=True)

In [10]:
tokenized_dataset = train_test.map(tokenizer_data, batched=True, remove_columns=['essay'])

Map:   0%|          | 0/3656 [00:00<?, ? examples/s]

Map:   0%|          | 0/914 [00:00<?, ? examples/s]

In [11]:
from datasets import ClassLabel

In [12]:
scores = ClassLabel(names=[str(i) for i in range(11)])

In [13]:
def map_labels(data):
    data['label'] = scores.str2int(str(data['score']))
    return data

In [14]:
tokenized_dataset = tokenized_dataset.map(map_labels, remove_columns=['score'])

Map:   0%|          | 0/3656 [00:00<?, ? examples/s]

Map:   0%|          | 0/914 [00:00<?, ? examples/s]

In [15]:
tokenized_dataset = tokenized_dataset.cast_column('label', scores)

Casting the dataset:   0%|          | 0/3656 [00:00<?, ? examples/s]

Casting the dataset:   0%|          | 0/914 [00:00<?, ? examples/s]

In [16]:
tokenized_dataset['train'].features['label']

ClassLabel(names=['0', '1', '2', '3', '4', '5', '6', '7', '8', '9', '10'])

In [17]:
# Importa as classes necessárias para carregar o tokenizador e o modelo de classificação
from transformers import AutoTokenizer, AutoModelForSequenceClassification

In [18]:
# Define os mapeamentos entre índice numérico e rótulo textual (notas de 0 a 10)
id2label = {i:str(i) for i in range(11)}
label2id = {value: key for key, value in id2label.items()}

In [19]:
# Carrega o tokenizador e o modelo pré-treinado mDeBERTa multilingual.
# O modelo original foi treinado para NLI com 3 classes; aqui substituímos
# a camada de classificação por uma nova com 11 saídas (notas 0-10).
# ignore_mismatched_sizes=True permite essa substituição sem erro.
# model_name = "MoritzLaurer/mDeBERTa-v3-base-xnli-multilingual-nli-2mil7"
model_name = base_model

tokenizer = AutoTokenizer.from_pretrained(model_name)

model = AutoModelForSequenceClassification.from_pretrained(
    model_name,
    num_labels=tokenized_dataset['train'].features['label'].num_classes,
    id2label=id2label,
    label2id=label2id,
    ignore_mismatched_sizes=True
)

Loading weights:   0%|          | 0/202 [00:00<?, ?it/s]

DebertaV2ForSequenceClassification LOAD REPORT from: Mel-Iza0/zero-shot
Key                             | Status     |                                                                                      
--------------------------------+------------+--------------------------------------------------------------------------------------
deberta.embeddings.position_ids | UNEXPECTED |                                                                                      
classifier.bias                 | MISMATCH   | Reinit due to size mismatch ckpt: torch.Size([3]) vs model:torch.Size([11])          
classifier.weight               | MISMATCH   | Reinit due to size mismatch ckpt: torch.Size([3, 768]) vs model:torch.Size([11, 768])

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISMATCH	:ckpt weights were loaded, but they did not match the original empty weight shapes.


In [20]:
# Define os hiperparâmetros de treinamento e cria o otimizador e o scheduler.
# AdamW é o otimizador padrão para fine-tuning de transformers.
# O scheduler reduz a taxa de aprendizado linearmente até zero ao longo do treino.
from torch.optim import AdamW
from transformers import get_linear_schedule_with_warmup

batch_size = 16
epochs = 2
batches_per_epoch = len(tokenized_dataset['train']) // batch_size
total_training_steps = int(batches_per_epoch * epochs)
learning_rate = 2e-5

optimizer = AdamW(model.parameters(), lr=learning_rate)

scheduler = get_linear_schedule_with_warmup(
    optimizer,
    num_warmup_steps=0,
    num_training_steps=total_training_steps
)

In [ ]:
import torch
torch.cuda.empty_cache()

# Configura e executa o fine-tuning do modelo usando a API Trainer do HuggingFace.
# O Trainer gerencia o loop de treinamento, avaliação por época e os logs automaticamente.
# use_cpu=True é necessário pois a GPU disponível (920MX) não é suportada pelo PyTorch 2.x.
from transformers import TrainingArguments, Trainer
from transformers import DataCollatorWithPadding

# DataCollatorWithPadding aplica padding dinâmico em cada batch
data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

training_args = TrainingArguments(
    output_dir="./results",
    num_train_epochs=epochs,
    per_device_train_batch_size=batch_size,
    per_device_eval_batch_size=batch_size,
    eval_strategy="epoch"
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset['train'],
    eval_dataset=tokenized_dataset['test'],
    data_collator=data_collator,
    optimizers=(optimizer, scheduler),
)

trainer.train()

Epoch,Training Loss,Validation Loss


In [ ]:
# Salva o modelo fine-tunado e o tokenizador em disco.
# Isso permite reutilizar o modelo em outros projetos sem retreinar.
model.save_pretrained("./essay-scorer")
tokenizer.save_pretrained("./essay-scorer")

In [ ]:
# Avalia o modelo nos dados de teste.
# trainer.predict() retorna os logits brutos; np.argmax seleciona a classe com maior pontuação.
# Exibe a acurácia geral e compara previsão vs. rótulo real nas primeiras 20 amostras.
import numpy as np

predictions = trainer.predict(tokenized_dataset['test'])
pred_labels = np.argmax(predictions.predictions, axis=1)
true_labels = predictions.label_ids

accuracy = (pred_labels == true_labels).mean()
print(f"Accuracy: {accuracy:.4f}")

print("\nPredicted vs True (first 20 samples):")
for pred, true in zip(pred_labels[:20], true_labels[:20]):
    match = "✓" if pred == true else "✗"
    print(f"  {match}  predicted={pred}  true={true}")